In [42]:
pip install pyserial

Note: you may need to restart the kernel to use updated packages.


In [43]:

import serial
print(serial.Serial)

import sys
sys.path.insert(0, "../../TOMBAK/base-folder/build/lib/aerodiode/")
from tombak import Tombak
from aerodiode import Aerodiode

%matplotlib inline
import pandas as pd

class pp():
    def __init__(self, port='/dev/serial/by-id/usb-FTDI_TTL-232R-3V3-AJ_FTCAVMZN-if00-port0', divider=8, width_ns=20):
        self.tombak = Tombak(port)

        self.tombak.set_status_instruction(self.tombak.INSTRUCT_FUNCTIONING_MODE, self.tombak.DIVIDER)
        self.tombak.set_status_instruction(self.tombak.INSTRUCT_PULSE_IN_SRC, self.tombak.DIRECT)
        self.tombak.set_status_instruction(self.tombak.INSTRUCT_TRIGGER_SRC, self.tombak.INT)
        self.tombak.set_status_instruction(self.tombak.INSTRUCT_SYNC_OUT_1_SRC, self.tombak.PULSE_OUT)

        self.set_output_pulse_width(width_ns)
        self.set_output_divider(divider)

        self.tombak.apply_all()

    def set_output_pulse_width(self, width_ns):
        """Set pulse width in nanoseconds."""
        self.tombak.set_integer_instruction(self.tombak.INSTRUCT_PULSE_OUT_WIDTH, int(width_ns))
        self.tombak.apply_all()

    def set_output_divider(self, div):
        """Set frequency divider."""
        self.divider = div
        self.tombak.set_integer_instruction(self.tombak.INSTRUCT_PULSE_IN_FREQUENCY_DIV, div)
        self.tombak.apply_all()

    def get_input_frequency(self):
        return self.tombak.measure_pulse_in_frequency()

    def get_output_frequency(self):
        return self.get_input_frequency() / self.divider

    def close(self):
        self.tombak.set_integer_instruction(self.tombak.INSTRUCT_PULSE_OUT_WIDTH, int(2**63 - 1))
        self.tombak.apply_all()

    def ten_mhz(self):
        input_freq = self.get_input_frequency()
        div = max(1, round(input_freq / 10e6))
        return self.set_output_divider(div)


<class 'serial.serialposix.Serial'>


In [66]:
device = pp(divider=280, width_ns=10)  # Set 20 ns width, divide by 8

In [67]:

device.set_output_pulse_width(10)
print("Input freq (Hz):", device.get_input_frequency())
print("Output freq (Hz):", device.get_output_frequency())


Input freq (Hz): 2033000
Output freq (Hz): 6350.0
